# Gradient Boosting Regressor — Thermal Predictor

Modelo de gradient boosting que prevê a temperatura de cada termopar (`tc_name`) de
uma `key_area` a partir das sete variáveis de configuração do ensaio.

**Um modelo por setor:** cada `key_area` recebe seu próprio modelo, treinado apenas
com os runs e os termopares daquela região — coerente com a H1 (Seção 4.2.3), em que
o efeito de uma configuração muda de sinal conforme a área.

**Entrada:** planilhas em formato wide geradas por `tratamento_final_dos_dados.ipynb`
(`data/<key_area>_data.csv`) — uma linha por run, uma coluna por termopar.

## Diferença estrutural em relação aos outros dois notebooks

`RandomForestRegressor` e `DecisionTreeRegressor` aceitam `Y` matricial nativamente:
uma folha guarda um vetor de médias, uma por termopar. **`GradientBoostingRegressor`
não** — ele exige `y` unidimensional e falha com
`ValueError: y should be a 1d array` se receber vários alvos.

A solução aqui é `MultiOutputRegressor`, que treina **um GBR independente por
termopar** e junta as previsões. Duas consequências práticas:

1. Os hiperparâmetros da grade levam o prefixo `estimator__`, porque são repassados
   ao GBR de dentro do wrapper.
2. Há uma importância de variáveis **por termopar**, e não uma única para o modelo
   inteiro — o que dá mais informação que o Random Forest, e é explorado na Célula 5.

Fora isso, a preparação dos dados e a avaliação estatística são idênticas às do
`Random_for_regre.ipynb`, para que os três modelos sejam comparáveis.

**Estrutura:** as Células 2 a 6 analisam **uma** área em detalhe — altere
`CAMINHO_PLANILHA` na Célula 2 para trocar de área. As Células 7 e 8 treinam as
**sete** de uma vez e comparam os resultados.

## 1. Bibliotecas e configuração global

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
SEMENTE = 42

## 2. Parâmetros e leitura da planilha

Formato wide: cada termopar é uma coluna-alvo independente. Tudo que não está em
`VARIAVEIS_INDEPENDENTES` é candidato a alvo.

Duas checagens são feitas na leitura, porque nem toda coluna extra é um termopar
válido: colunas **sem nome** (`Unnamed: N`, geradas quando o `tc_name` original
estava em branco — é o caso de `ct_knobs_data.csv`) e colunas **inteiramente
vazias** são descartadas com aviso, em vez de entrarem como alvo do modelo.

In [2]:
# ============================================================
# CÉLULA 2 — PARÂMETROS E LEITURA DA PLANILHA
# Formato wide: cada termopar é uma coluna-alvo independente
# ============================================================

# Troque aqui para analisar outra key_area em detalhe:
#   ../data/mainback_data.csv        ../data/ipc_comp_data.csv
#   ../data/coil_temps_data.csv      ../data/ipc_cooling_data.csv
#   ../data/main_vent_sys_data.csv   ../data/oven_hmi_data.csv
#   ../data/ct_knobs_data.csv
CAMINHO_PLANILHA = "../data/mainback_data.csv"

VARIAVEIS_INDEPENDENTES = [
    "blower_direction",
    "housing",
    "board_position",
    "run_case",
    "losses",
    "deflector_hole",
    "material_coil_support",
]

# Um termopar só entra como alvo se tiver leitura em pelo menos esta fração dos
# runs. Sensores abaixo disso são removidos como COLUNA — descartá-los custa um
# alvo, enquanto exigir que estejam presentes custaria quase todos os runs.
COBERTURA_MINIMA_ALVO = 0.50

# Número de folds do GridSearchCV. É reduzido automaticamente se o conjunto de
# treino for pequeno demais (ver Célula 5).
N_FOLDS_DESEJADO = 3


def identificar_alvos(df, independentes):
    """Separa as colunas-alvo válidas das colunas que não representam termopar."""
    alvos, descartadas = [], []
    for coluna in df.columns:
        if coluna in independentes:
            continue
        if str(coluna).startswith("Unnamed:") or not str(coluna).strip():
            descartadas.append((coluna, "coluna sem nome de termopar"))
        elif df[coluna].isna().all():
            descartadas.append((coluna, "coluna inteiramente vazia"))
        else:
            alvos.append(coluna)
    return alvos, descartadas


def resumir_alvos(df, alvos):
    """Tabela de cobertura e estatística descritiva de cada termopar."""
    return pd.DataFrame([
        {
            "termopar": alvo,
            "leituras": int(df[alvo].notna().sum()),
            "cobertura": df[alvo].notna().mean(),
            "min": df[alvo].min(),
            "max": df[alvo].max(),
            "média": df[alvo].mean(),
            "desvio": df[alvo].std(),
        }
        for alvo in alvos
    ]).sort_values("cobertura", ascending=False, ignore_index=True)


def listar_alvos_duplicados(df, alvos):
    """Pares de termopares com leitura idêntica (possível duplicação de sensor)."""
    return [
        (alvo_a, alvo_b)
        for i, alvo_a in enumerate(alvos)
        for alvo_b in alvos[i + 1:]
        if df[alvo_a].equals(df[alvo_b])
    ]


dados = pd.read_csv(CAMINHO_PLANILHA)
VARIAVEIS_ALVO, COLUNAS_DESCARTADAS = identificar_alvos(dados, VARIAVEIS_INDEPENDENTES)

print(f"Arquivo: {CAMINHO_PLANILHA}")
print(f"Linhas: {len(dados)} | Colunas: {dados.shape[1]}")

if COLUNAS_DESCARTADAS:
    print("\nColunas ignoradas (não são termopares válidos):")
    for coluna, motivo in COLUNAS_DESCARTADAS:
        print(f"   {coluna!r}: {motivo}")

print(f"\nTermopares candidatos ({len(VARIAVEIS_ALVO)}):")
print(resumir_alvos(dados, VARIAVEIS_ALVO).round(2).to_string(index=False))

print(f"\nValores nulos nos alvos: {int(dados[VARIAVEIS_ALVO].isna().sum().sum())}")

duplicadas = listar_alvos_duplicados(dados, VARIAVEIS_ALVO)
if duplicadas:
    print("\nATENÇÃO — colunas-alvo com valores idênticos:")
    for alvo_a, alvo_b in duplicadas:
        print(f"   {alvo_a}  ==  {alvo_b}")

dados.head()

Arquivo: ../data/mainback_data.csv
Linhas: 56 | Colunas: 17

Termopares candidatos (10):
                          termopar  leituras  cobertura   min    max  média  desvio
       ACU Ambient (Hill Only) Top        56       1.00 43.01  99.06  62.43   13.33
ACU K8 Housing / Double Line Break        56       1.00 41.74  73.22  50.98    9.45
      Ambient Below Terminal Block        56       1.00 37.31  80.53  48.83   11.79
             Convect Motor Winding        56       1.00 64.00 112.05  85.76   11.28
Door Lock Motor Ambient 1/2" Below        56       1.00 75.41 120.45  91.24    8.09
              L2, Wire, Power Cord        56       1.00 41.38  81.23  52.70   11.84
      Storage Drawer (Metal Above)        56       1.00 90.25 171.57 105.37   22.21
        Door Lock Microswitch Body        55       0.98 83.77 120.70  95.82    7.84
        Hot Surface Indicator Body        55       0.98 37.30  60.98  44.22    5.95
           Power On Indicator Body        55       0.98 35.41  60.73  4

,blower_direction,housing,board_position,run_case,losses,deflector_hole,material_coil_support,ACU Ambient (Hill Only) Top,ACU K8 Housing / Double Line Break,Ambient Below Terminal Block,Convect Motor Winding,Door Lock Microswitch Body,"Door Lock Motor Ambient 1/2"" Below",Hot Surface Indicator Body,"L2, Wire, Power Cord",Power On Indicator Body,Storage Drawer (Metal Above)
0,To back,Closed,Center,Broil C2,Simone Losses,Open,Aluminium,58.07,45.77,45.37,81.95,99.11,90.74,45.22,47.72,43.02,96.11
1,To back,Closed,Center,Broil C2,Simone Modified - Back Order,Closed,Aluminium,64.03,48.76,46.65,87.91,105.68,98.05,44.29,50.57,42.57,100.54
2,To back,Closed,Center,Broil Qualitative,Simone Losses,Open,Aluminium,57.59,45.23,42.85,81.04,98.17,90.78,43.06,45.93,44.24,95.95
3,To back,Open,Center,Broil C2,Simone Losses,Open,Aluminium,81.68,68.86,73.74,103.28,84.49,75.41,45.04,77.65,41.74,96.88
4,To back,Open,Center,Broil C2,Simone Losses,Open,Aluminium,88.54,72.72,80.53,104.92,87.46,78.76,43.96,81.23,41.44,97.16


## 3. Seleção de alvos, padronização das categorias e codificação

Primeiro descarta o **sensor** de cobertura baixa (coluna), depois descarta as
**linhas** que ainda tenham buraco entre os alvos que sobraram. A ordem importa:
fazer `dropna` direto sobre todos os alvos é *listwise deletion* — basta um sensor
pouco medido para eliminar quase todos os runs. Em `main_vent_sys_data.csv`, dois
sensores medidos em apenas 4 dos 56 runs derrubariam a base de 56 para 4 linhas.

Idêntico ao `Random_for_regre.ipynb`, de propósito: as três abordagens precisam
partir exatamente da mesma base para que a comparação de métricas faça sentido.

In [3]:
# ============================================================
# CÉLULA 3 — SELEÇÃO DE ALVOS, PADRONIZAÇÃO E CODIFICAÇÃO ORDINAL
# ============================================================
GRAFIAS_DIVERGENTES = {
    "blower_direction": {"To back": "To Back", "To front": "To Front"},
    "losses":           {"No losses": "No Losses"},
    "run_case":         {"Broil Case 2": "Broil C2", "Broil Qual": "Broil Qualitative", "Qualitative": "Broil Qualitative"},
    "housing":          {"Open - Updated": "Open"}
}


def filtrar_alvos_por_cobertura(df, alvos, cobertura_minima):
    """Mantém apenas os termopares medidos em uma fração mínima dos runs."""
    mantidos, removidos = [], []
    for alvo in alvos:
        cobertura = df[alvo].notna().mean()
        if cobertura >= cobertura_minima:
            mantidos.append(alvo)
        else:
            removidos.append((alvo, int(df[alvo].notna().sum()), cobertura))
    return mantidos, removidos


def padronizar_categorias(df, independentes, correcoes_por_variavel):
    """Unifica grafias divergentes e transforma nulos em categoria explícita."""
    df = df.copy()
    for variavel, correcoes in correcoes_por_variavel.items():
        if variavel in df.columns:
            df[variavel] = df[variavel].replace(correcoes)
    for variavel in independentes:
        df[variavel] = df[variavel].fillna("Nao Informado").astype(str)
    return df


def codificar_ordinal(df, independentes):
    """Codificação ordinal (0, 1, 2, ...) em ordem alfabética por variável."""
    X = df[independentes].copy()
    dicionario = {}
    for variavel in independentes:
        codigos = {categoria: i for i, categoria in enumerate(sorted(X[variavel].unique()))}
        dicionario[variavel] = codigos
        X[variavel] = X[variavel].map(codigos)
    return X, dicionario


# 1) Descarta o SENSOR pouco medido (coluna), não o run (linha).
VARIAVEIS_ALVO, alvos_removidos = filtrar_alvos_por_cobertura(
    dados, VARIAVEIS_ALVO, COBERTURA_MINIMA_ALVO
)

if alvos_removidos:
    print(f"Termopares removidos por cobertura < {COBERTURA_MINIMA_ALVO:.0%}:")
    for alvo, leituras, cobertura in alvos_removidos:
        print(f"   {alvo:40s} {leituras:3d}/{len(dados)} runs ({cobertura:.1%})")
    print()

if not VARIAVEIS_ALVO:
    raise ValueError(
        f"Nenhum termopar atingiu a cobertura mínima de {COBERTURA_MINIMA_ALVO:.0%}. "
        f"Reduza COBERTURA_MINIMA_ALVO na Célula 2 para modelar este arquivo."
    )

# 2) Só agora descarta as LINHAS com buraco entre os alvos que sobraram.
dados_tratados = dados.dropna(subset=VARIAVEIS_ALVO).copy()

linhas_removidas = len(dados) - len(dados_tratados)
if linhas_removidas:
    print(f"Linhas removidas por alvo ausente: {linhas_removidas}\n")

dados_tratados = padronizar_categorias(
    dados_tratados, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES
)
X, dicionario_codificacao = codificar_ordinal(dados_tratados, VARIAVEIS_INDEPENDENTES)
Y = dados_tratados[VARIAVEIS_ALVO]

print("Dicionário de codificação:\n")
for variavel, codigos in dicionario_codificacao.items():
    print(f"{variavel} ({len(codigos)} categorias)")
    for categoria, codigo in codigos.items():
        print(f"   {codigo:>2d} = {categoria}")
    print()

print(f"Matriz de features: {X.shape}")
print(f"Colunas-alvo: {len(VARIAVEIS_ALVO)}")
print(f"Linhas úteis: {len(dados_tratados)} de {len(dados)}")
print(f"Combinações distintas de configuração: {X.drop_duplicates().shape[0]}")

Linhas removidas por alvo ausente: 2

Dicionário de codificação:

blower_direction (2 categorias)
    0 = To Back
    1 = To Front

housing (2 categorias)
    0 = Closed
    1 = Open

board_position (2 categorias)
    0 = Center
    1 = Right

run_case (4 categorias)
    0 = Broil C1
    1 = Broil C2
    2 = Broil Qualitative
    3 = Pyro

losses (14 categorias)
    0 = Broil C2 Simone - 240/145/180/180
    1 = Broil C2 Simone - Front Order
    2 = Broil Miro - Front Order
    3 = C1 Simone - 240/145/180/180
    4 = Miro
    5 = Miro - Front Order
    6 = Miro Qualitative  - Back Order
    7 = Miro Qualitative  - Front Order
    8 = No Losses
    9 = Não informado
   10 = Qual Simone - Front Order
   11 = Simone
   12 = Simone Losses
   13 = Simone Modified - Back Order

deflector_hole (2 categorias)
    0 = Closed
    1 = Open

material_coil_support (3 categorias)
    0 = Aluminium 
    1 = Aluminium Plate
    2 = Plastic 

Matriz de features: (54, 7)
Colunas-alvo: 10
Linhas úteis: 54

## 4. Separação treino (70%) / teste (20%) / verificação (10%)

`Y` é matricial: uma coluna por termopar, lida de `dados_tratados` para permanecer
alinhada com as linhas de `X`. O `MultiOutputRegressor` da Célula 5 se encarrega de
quebrar essa matriz em um problema por termopar.

A célula checa antes da divisão se sobraram linhas suficientes para formar os três
conjuntos e ainda alimentar a validação cruzada, falhando com uma mensagem que diz
o que ajustar — em vez do `ValueError` genérico do `GridSearchCV`.

In [4]:
# ============================================================
# CÉLULA 4 — SEPARAÇÃO TREINO (70%) / TESTE (20%) / VERIFICAÇÃO (10%)
# Y é matricial (uma coluna por termopar)
# ============================================================
PROPORCAO_TREINO = 0.70
PROPORCAO_TESTE = 0.20
PROPORCAO_VERIFICACAO = 0.10

MINIMO_DE_LINHAS = 10
if len(X) < MINIMO_DE_LINHAS:
    raise ValueError(
        f"Apenas {len(X)} linhas úteis após o tratamento — insuficiente para dividir em "
        f"treino/teste/verificação e rodar validação cruzada. Verifique a Célula 3: "
        f"provavelmente há termopares esparsos derrubando linhas. Considere reduzir "
        f"COBERTURA_MINIMA_ALVO (hoje {COBERTURA_MINIMA_ALVO:.0%}) na Célula 2."
    )

X_treino, X_restante, Y_treino, Y_restante = train_test_split(
    X, Y,
    train_size=PROPORCAO_TREINO,
    random_state=SEMENTE,
)

proporcao_verificacao_no_restante = (
    PROPORCAO_VERIFICACAO / (PROPORCAO_TESTE + PROPORCAO_VERIFICACAO)
)

X_teste, X_verificacao, Y_teste, Y_verificacao = train_test_split(
    X_restante, Y_restante,
    test_size=proporcao_verificacao_no_restante,
    random_state=SEMENTE,
)

print(f"Treino:       {len(X_treino):3d} linhas ({len(X_treino)/len(X):.1%})")
print(f"Teste:        {len(X_teste):3d} linhas ({len(X_teste)/len(X):.1%})")
print(f"Verificação:  {len(X_verificacao):3d} linhas ({len(X_verificacao)/len(X):.1%})")
print(f"\nFormato de X: {X_treino.shape}  |  Formato de Y: {Y_treino.shape}")
print(f"Modelos que serão treinados nesta área: {Y_treino.shape[1]} "
      f"(um GBR por termopar)")

Treino:        37 linhas (68.5%)
Teste:         11 linhas (20.4%)
Verificação:    6 linhas (11.1%)

Formato de X: (37, 7)  |  Formato de Y: (37, 10)
Modelos que serão treinados nesta área: 10 (um GBR por termopar)


## 5. Treinamento do modelo (Gradient Boosting, um GBR por termopar)

`MultiOutputRegressor` envolve o `GradientBoostingRegressor` e treina um regressor
independente por termopar — é o que torna o GBR utilizável neste dataset multi-alvo.
Por isso os parâmetros da grade levam o prefixo `estimator__`.

A grade é a de controle de superajuste que interessa em boosting: `learning_rate` e
`n_estimators` (que se compensam — taxa menor exige mais estágios), `max_depth`
raso (2 ou 3, o padrão em boosting é somar muitos aprendizes fracos, não poucos
fortes) e `subsample` < 1 para introduzir estocasticidade, útil com ~40 linhas.

`n_jobs` fica no `GridSearchCV`, e o `MultiOutputRegressor` roda em série, para não
disputar núcleos entre os dois níveis de paralelismo.

**Custo:** esta é a célula mais lenta dos três notebooks — 16 combinações × 3 folds ×
um GBR por termopar. Em `ipc_comp_data.csv` (37 termopares) leva cerca de 80 s.

In [5]:
# ============================================================
# CÉLULA 5 — TREINAMENTO (GRADIENT BOOSTING, UM GBR POR TERMOPAR)
# GradientBoostingRegressor não aceita Y matricial: MultiOutputRegressor
# treina um regressor independente para cada termopar.
# ============================================================
GRADE_HIPERPARAMETROS = {
    "estimator__n_estimators":  [100, 300],
    "estimator__learning_rate": [0.05, 0.1],
    "estimator__max_depth":     [2, 3],
    "estimator__subsample":     [1.0, 0.8],
}

# Nunca pedir mais folds do que há linhas de treino.
n_folds = max(2, min(N_FOLDS_DESEJADO, len(X_treino)))
if n_folds < N_FOLDS_DESEJADO:
    print(f"Aviso: {len(X_treino)} linhas de treino — usando {n_folds} folds "
          f"em vez de {N_FOLDS_DESEJADO}.\n")

busca = GridSearchCV(
    estimator=MultiOutputRegressor(
        GradientBoostingRegressor(random_state=SEMENTE), n_jobs=1
    ),
    param_grid=GRADE_HIPERPARAMETROS,
    cv=n_folds,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
busca.fit(X_treino, Y_treino)

modelo = busca.best_estimator_

print("Melhores hiperparâmetros:")
for parametro, valor in busca.best_params_.items():
    print(f"   {parametro.replace('estimator__', '')}: {valor}")

print(f"\nRegressores treinados: {len(modelo.estimators_)} (um por termopar)")


def importancias_por_alvo(modelo_multioutput, features, alvos):
    """Matriz termopar x variável — cada GBR interno tem sua própria importância."""
    return pd.DataFrame(
        [estimador.feature_importances_ for estimador in modelo_multioutput.estimators_],
        index=alvos,
        columns=features,
    )


matriz_importancia = importancias_por_alvo(modelo, VARIAVEIS_INDEPENDENTES, VARIAVEIS_ALVO)
importancia = matriz_importancia.mean().sort_values(ascending=False)

print(f"\nImportância média das variáveis (entre os {len(VARIAVEIS_ALVO)} termopares):")
print(importancia.round(4).to_string())



print("Termopar mais sensível a cada variável:")
for variavel in importancia.index:
    alvo = matriz_importancia[variavel].idxmax()
    print(f"   {variavel:24s} -> {alvo} ({matriz_importancia.loc[alvo, variavel]:.3f})")

Melhores hiperparâmetros:
   learning_rate: 0.1
   max_depth: 3
   n_estimators: 100
   subsample: 0.8

Regressores treinados: 10 (um por termopar)

Importância média das variáveis (entre os 10 termopares):
run_case                 0.4993
housing                  0.1443
blower_direction         0.1337
board_position           0.0955
deflector_hole           0.0651
losses                   0.0593
material_coil_support    0.0028
Termopar mais sensível a cada variável:
   run_case                 -> Storage Drawer (Metal Above) (0.893)
   housing                  -> ACU K8 Housing / Double Line Break (0.259)
   blower_direction         -> Ambient Below Terminal Block (0.333)
   board_position           -> ACU Ambient (Hill Only) Top (0.304)
   deflector_hole           -> ACU Ambient (Hill Only) Top (0.164)
   losses                   -> Ambient Below Terminal Block (0.107)
   material_coil_support    -> Door Lock Microswitch Body (0.012)


## 6. Avaliação estatística por termopar

MAE, MSE, RMSE, R² e MAPE calculados coluna a coluna nos três conjuntos, seguidos
da média entre termopares e da fração de previsões dentro da margem de 10%
acordada com o parceiro. Idêntica à do `Random_for_regre.ipynb`.

In [6]:
# ============================================================
# CÉLULA 6 — AVALIAÇÃO ESTATÍSTICA POR TERMOPAR
# ============================================================
def avaliar_multioutput(Y_real, Y_previsto, nome_conjunto, alvos):
    """Calcula métricas para cada coluna-alvo separadamente."""
    linhas = []
    for indice, termopar in enumerate(alvos):
        real = Y_real.iloc[:, indice].values
        previsto = Y_previsto[:, indice]
        mse = mean_squared_error(real, previsto)
        linhas.append({
            "conjunto": nome_conjunto,
            "termopar": termopar,
            "MAE":      mean_absolute_error(real, previsto),
            "MSE":      mse,
            "RMSE":     np.sqrt(mse),
            "R2":       r2_score(real, previsto),
            "MAPE_%":   np.mean(np.abs((real - previsto) / real)) * 100,
        })
    return linhas


previsoes_treino      = modelo.predict(X_treino)
previsoes_teste       = modelo.predict(X_teste)
previsoes_verificacao = modelo.predict(X_verificacao)

avaliacao = pd.DataFrame(
    avaliar_multioutput(Y_treino,      previsoes_treino,      "Treino",      VARIAVEIS_ALVO)
  + avaliar_multioutput(Y_teste,       previsoes_teste,       "Teste",       VARIAVEIS_ALVO)
  + avaliar_multioutput(Y_verificacao, previsoes_verificacao, "Verificação", VARIAVEIS_ALVO)
)

print("=" * 100)
print(f"DESEMPENHO POR TERMOPAR — {CAMINHO_PLANILHA}")
print("=" * 100)
print(avaliacao.round(4).to_string(index=False))

print("\n" + "=" * 100)
print("MÉDIA ENTRE OS TERMOPARES")
print("=" * 100)
resumo_conjuntos = (
    avaliacao.groupby("conjunto", sort=False)[["MAE", "MSE", "RMSE", "R2", "MAPE_%"]]
             .mean()
)
print(resumo_conjuntos.round(4).to_string())

erro_percentual_teste = np.abs((Y_teste.values - previsoes_teste) / Y_teste.values)
print(f"\nPrevisões dentro da margem de 10% (teste, todos os termopares): "
      f"{(erro_percentual_teste <= 0.10).mean():.1%}")

# Gap treino -> teste: leitura direta de superajuste (risco R03 da Seção 4.1.5).
gap = resumo_conjuntos.loc["Teste", "MAE"] - resumo_conjuntos.loc["Treino", "MAE"]
print(f"Gap de MAE treino -> teste: {gap:+.2f} °C")

DESEMPENHO POR TERMOPAR — ../data/mainback_data.csv
   conjunto                           termopar     MAE      MSE    RMSE      R2  MAPE_%
     Treino        ACU Ambient (Hill Only) Top  1.8875  16.7832  4.0967  0.8818  3.0153
     Treino ACU K8 Housing / Double Line Break  1.6067  15.9892  3.9986  0.7219  2.9673
     Treino       Ambient Below Terminal Block  2.1821  22.5634  4.7501  0.7447  4.3381
     Treino              Convect Motor Winding  2.4048  34.2234  5.8501  0.6261  2.7553
     Treino         Door Lock Microswitch Body  1.2508   8.6545  2.9418  0.8129  1.2674
     Treino Door Lock Motor Ambient 1/2" Below  1.7093  18.3283  4.2812  0.6173  1.7507
     Treino         Hot Surface Indicator Body  1.2409  10.1141  3.1803  0.6456  2.7367
     Treino               L2, Wire, Power Cord  1.8578  20.0985  4.4831  0.7717  3.3097
     Treino            Power On Indicator Body  1.2311   9.0034  3.0006  0.5331  2.8300
     Treino       Storage Drawer (Metal Above)  2.9333  85.6039  9.2

## 7. Um modelo por setor — treino das sete áreas

Encadeia as funções das células anteriores em um único pipeline e treina um conjunto
independente de GBRs para cada `key_area`. Os modelos ficam guardados em
`MODELOS_POR_AREA`, junto com os alvos e o dicionário de codificação de cada um — o
conjunto necessário para servir previsões no dashboard.

Esta célula treina, ao todo, um GBR para cada termopar de cada área (74 no total nas
sete planilhas), cada um com busca de hiperparâmetros. Leva alguns minutos.

In [7]:
# ============================================================
# CÉLULA 7 — UM MODELO POR SETOR (KEY_AREA)
# ============================================================
PLANILHAS = [
    "../data/mainback_data.csv",
    "../data/coil_temps_data.csv",
    "../data/main_vent_sys_data.csv",
    "../data/ct_knobs_data.csv",
    "../data/ipc_comp_data.csv",
    "../data/ipc_cooling_data.csv",
    "../data/oven_hmi_data.csv",
]


def preparar_area(caminho, cobertura_minima=COBERTURA_MINIMA_ALVO):
    """Leitura + seleção de alvos + codificação de uma planilha. Devolve X, Y e metadados."""
    df = pd.read_csv(caminho)
    alvos, _ = identificar_alvos(df, VARIAVEIS_INDEPENDENTES)
    alvos, removidos = filtrar_alvos_por_cobertura(df, alvos, cobertura_minima)
    if not alvos:
        raise ValueError(f"{caminho}: nenhum termopar com cobertura >= {cobertura_minima:.0%}")

    tratado = df.dropna(subset=alvos)
    tratado = padronizar_categorias(tratado, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES)
    X_area, dicionario = codificar_ordinal(tratado, VARIAVEIS_INDEPENDENTES)
    return X_area, tratado[alvos], alvos, dicionario, removidos


def treinar_area(caminho, n_folds_desejado=N_FOLDS_DESEJADO):
    """Treina os GBRs de uma key_area e devolve o modelo com suas métricas."""
    X_area, Y_area, alvos, dicionario, removidos = preparar_area(caminho)

    if len(X_area) < MINIMO_DE_LINHAS:
        raise ValueError(f"{caminho}: apenas {len(X_area)} linhas úteis")

    X_tr, X_resto, Y_tr, Y_resto = train_test_split(
        X_area, Y_area, train_size=PROPORCAO_TREINO, random_state=SEMENTE)
    X_te, X_ve, Y_te, Y_ve = train_test_split(
        X_resto, Y_resto,
        test_size=PROPORCAO_VERIFICACAO / (PROPORCAO_TESTE + PROPORCAO_VERIFICACAO),
        random_state=SEMENTE)

    modelo_area = GridSearchCV(
        estimator=MultiOutputRegressor(
            GradientBoostingRegressor(random_state=SEMENTE), n_jobs=1
        ),
        param_grid=GRADE_HIPERPARAMETROS,
        cv=max(2, min(n_folds_desejado, len(X_tr))),
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    ).fit(X_tr, Y_tr).best_estimator_

    metricas = pd.DataFrame(
        avaliar_multioutput(Y_tr, modelo_area.predict(X_tr), "Treino", alvos)
      + avaliar_multioutput(Y_te, modelo_area.predict(X_te), "Teste", alvos)
      + avaliar_multioutput(Y_ve, modelo_area.predict(X_ve), "Verificação", alvos)
    )
    medias = metricas.groupby("conjunto", sort=False)[["MAE", "RMSE", "R2", "MAPE_%"]].mean()
    erro_pct = np.abs((Y_te.values - modelo_area.predict(X_te)) / Y_te.values)

    resumo = {
        "setor": caminho.split("/")[-1].replace("_data.csv", ""),
        "linhas": len(X_area),
        "alvos": len(alvos),
        "alvos_removidos": len(removidos),
        "MAE_treino": medias.loc["Treino", "MAE"],
        "MAE_teste": medias.loc["Teste", "MAE"],
        "RMSE_teste": medias.loc["Teste", "RMSE"],
        "R2_teste": medias.loc["Teste", "R2"],
        "MAPE_teste_%": medias.loc["Teste", "MAPE_%"],
        "dentro_de_10%": (erro_pct <= 0.10).mean(),
    }
    detalhe = {
        "modelo": modelo_area,
        "alvos": alvos,
        "codificacao": dicionario,
        "metricas": metricas,
        "importancias": importancias_por_alvo(modelo_area, VARIAVEIS_INDEPENDENTES, alvos),
    }
    return resumo, detalhe


MODELOS_POR_AREA = {}
resumos = []

for caminho in PLANILHAS:
    try:
        resumo, detalhe = treinar_area(caminho)
        MODELOS_POR_AREA[resumo["setor"]] = detalhe
        resumos.append(resumo)
        print(f"OK    {resumo['setor']:15s} "
              f"{resumo['linhas']:3d} linhas, {resumo['alvos']:2d} GBRs treinados")
    except Exception as erro:
        print(f"FALHA {caminho}: {type(erro).__name__}: {erro}")

total_gbrs = sum(r["alvos"] for r in resumos)
print(f"\n{len(MODELOS_POR_AREA)} setores modelados, {total_gbrs} GBRs no total: "
      f"{list(MODELOS_POR_AREA)}")

OK    mainback         54 linhas, 10 GBRs treinados


OK    coil_temps       49 linhas,  8 GBRs treinados


OK    main_vent_sys    56 linhas,  5 GBRs treinados


OK    ct_knobs         55 linhas,  6 GBRs treinados


OK    ipc_comp         24 linhas, 37 GBRs treinados


OK    ipc_cooling      53 linhas,  4 GBRs treinados


OK    oven_hmi         55 linhas,  4 GBRs treinados

7 setores modelados, 74 GBRs no total: ['mainback', 'coil_temps', 'main_vent_sys', 'ct_knobs', 'ipc_comp', 'ipc_cooling', 'oven_hmi']


## 8. Comparativo entre setores

Métricas médias de cada setor no seu próprio conjunto de teste. O par
`MAE_treino` / `MAE_teste` mede o superajuste — o risco R03 da Seção 4.1.5.

Os valores não são comparáveis entre setores em termos absolutos: cada área tem seu
próprio patamar de temperatura (de ~47 °C em `CT Knobs` a ~124 °C em `Coil Temps`),
então o MAPE e a fração dentro de 10% são as colunas que permitem comparação justa.

In [8]:
# ============================================================
# CÉLULA 8 — COMPARATIVO ENTRE SETORES
# ============================================================
comparativo = pd.DataFrame(resumos)

print("=" * 125)
print("COMPARATIVO ENTRE SETORES — Gradient Boosting, métricas médias no conjunto de teste")
print("=" * 125)
print(comparativo.round(4).to_string(index=False))

print("\n" + "=" * 125)
print("MÉDIA ENTRE OS SETORES")
print("=" * 125)
print(comparativo[["MAE_teste", "RMSE_teste", "R2_teste", "MAPE_teste_%", "dentro_de_10%"]]
      .mean().round(4).to_string())



COMPARATIVO ENTRE SETORES — Gradient Boosting, métricas médias no conjunto de teste
        setor  linhas  alvos  alvos_removidos  MAE_treino  MAE_teste  RMSE_teste  R2_teste  MAPE_teste_%  dentro_de_10%
     mainback      54     10                0      1.8304     5.3119      8.8517   -0.0534        8.8374         0.7455
   coil_temps      49      8                0      3.2749     7.3362     11.9516    0.8653        6.8927         0.7625
main_vent_sys      56      5                2      2.8327     3.2842      5.2150    0.5626        3.7887         0.8727
     ct_knobs      55      6                0      2.3664     1.7356      2.4092    0.7518        3.5293         0.9394
     ipc_comp      24     37                0      1.3371     4.5813      5.0857   -1.0411        6.5422         0.8757
  ipc_cooling      53      4                0      1.7521     4.6483      8.5954    0.0501        6.2456         0.7750
     oven_hmi      55      4                0      1.3669     2.2436      2.